# Student Performance Analysis
**Name:** Your Name  
**Roll No:** Your Roll Number  
**Date:** 2025  
**Dataset:** Students Performance in Exams  

---
## Common Setup — Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set global plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load raw dataset
df = pd.read_csv('data/StudentsPerformance.csv')

print('Dataset loaded successfully!')
print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Basic dataset info
print('=== Dataset Info ===')
df.info()
print('\n=== Column Names ===')
print(df.columns.tolist())
print('\n=== Data Types ===')
print(df.dtypes)

---
# TASK 2 — Descriptive Statistics

## Task 2.1 — Mean, Median, Mode, Std, Min, Max

In [ ]:
scores = ['math score', 'reading score', 'writing score']

print('='*55)
print('         DESCRIPTIVE STATISTICS FOR EXAM SCORES')
print('='*55)

for col in scores:
    print(f'\n--- {col.upper()} ---')
    print(f'  Mean   : {df[col].mean():.2f}')
    print(f'  Median : {df[col].median():.2f}')
    print(f'  Mode   : {df[col].mode()[0]}')
    print(f'  Std Dev: {df[col].std():.2f}')
    print(f'  Min    : {df[col].min()}')
    print(f'  Max    : {df[col].max()}')

In [ ]:
# Summary table using describe()
print('\n=== Summary Statistics Table ===')
df[scores].describe().round(2)

## Task 2.2 — Mean vs Median Comparison

In [ ]:
print('='*55)
print('           MEAN vs MEDIAN COMPARISON')
print('='*55)
print(f'{"Score":<20} {"Mean":>8} {"Median":>8} {"Diff":>8} {"Skew":>10}')
print('-'*55)

for col in scores:
    mean   = df[col].mean()
    median = df[col].median()
    diff   = mean - median
    skew   = df[col].skew()
    print(f'{col:<20} {mean:>8.2f} {median:>8.2f} {diff:>8.2f} {skew:>10.4f}')

## Task 2.3 — GroupBy Analysis

In [ ]:
# Average scores by gender
print('=== Average Scores by Gender ===')
gender_stats = df.groupby('gender')[scores].mean().round(2)
print(gender_stats)
print()

# Average scores by parental level of education
print('=== Average Scores by Parental Level of Education ===')
edu_stats = df.groupby('parental level of education')[scores].mean().round(2)
print(edu_stats)

In [ ]:
# Additional groupby — test prep course
print('=== Average Scores by Test Preparation Course ===')
prep_stats = df.groupby('test preparation course')[scores].mean().round(2)
print(prep_stats)

print()
print('=== Average Scores by Lunch Type ===')
lunch_stats = df.groupby('lunch')[scores].mean().round(2)
print(lunch_stats)

## Task 2.4 — Observations

**Observation 1:** Female students outperform male students in both reading (avg ~72.6 vs ~65.5) and writing (avg ~72.5 vs ~63.3), while male students score slightly higher in math (avg ~68.7 vs ~63.6).

**Observation 2:** Parental education level has a clear positive correlation with student performance. Students whose parents hold a master's degree score the highest across all three subjects.

**Observation 3:** The Mean and Median values are very close for all three scores, suggesting the distributions are approximately symmetric (not heavily skewed).

**Observation 4:** Students who completed the test preparation course scored significantly higher than those who did not — approximately 5–10 points more on average across all subjects.

**Observation 5:** Students with a standard lunch (vs free/reduced) consistently score higher in all subjects, possibly reflecting socioeconomic factors influencing academic performance.

---
# TASK 3 — Data Cleaning

## Task 3.1 — Create Messy Dataset

In [ ]:
# Create a messy version of the dataset
df_raw = pd.read_csv('data/StudentsPerformance.csv')
messy = df_raw.copy()

np.random.seed(42)

# Problem 1: Add missing values
idx_nan = np.random.choice(messy.index, 40, replace=False)
messy.loc[idx_nan[:15], 'math score']    = np.nan
messy.loc[idx_nan[15:28], 'reading score'] = np.nan
messy.loc[idx_nan[28:], 'writing score']   = np.nan

# Problem 2: Add duplicate rows
duplicate_rows = messy.sample(25, random_state=1)
messy = pd.concat([messy, duplicate_rows], ignore_index=True)

# Problem 3: Add outliers (impossible scores)
messy.loc[5,  'math score']    = 150   # above 100
messy.loc[10, 'reading score'] = -5    # below 0
messy.loc[15, 'writing score'] = 999   # absurd value

# Problem 4: Add invalid category values
messy.loc[3, 'gender'] = 'unknown'
messy.loc[7, 'gender'] = 'MALE'       # wrong case
messy.loc[20, 'lunch'] = 'N/A'

# Problem 5: Add whitespace in string columns
messy.loc[1, 'gender'] = ' female '
messy.loc[2, 'gender'] = ' male'

# Save messy file
messy.to_csv('data/messy_students.csv', index=False)

print('Messy dataset created and saved!')
print('Messy shape:', messy.shape)

## Task 3.2 — Show All Problems

In [ ]:
df_messy = pd.read_csv('data/messy_students.csv')

print('=== MESSY DATASET INFO ===')
print('Shape:', df_messy.shape)
df_messy.info()

In [ ]:
print('=== MISSING VALUES ===')
missing = df_messy.isnull().sum()
missing_pct = (missing / len(df_messy) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
print('=== DUPLICATE ROWS ===')
print('Total duplicates:', df_messy.duplicated().sum())

print('\n=== INVALID CATEGORIES ===')
print('Gender values:', df_messy['gender'].unique())
print('Lunch values: ', df_messy['lunch'].unique())

print('\n=== OUTLIERS IN SCORES ===')
for col in ['math score', 'reading score', 'writing score']:
    out_low  = df_messy[col].dropna()[df_messy[col].dropna() < 0]
    out_high = df_messy[col].dropna()[df_messy[col].dropna() > 100]
    if len(out_low) > 0 or len(out_high) > 0:
        print(f'  {col}: Below 0 → {list(out_low.values)}  |  Above 100 → {list(out_high.values)}')

## Task 3.3 — Clean Step by Step

In [ ]:
# ---- STEP 1: Fix whitespace in string columns ----
df_clean = df_messy.copy()

str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()

print('STEP 1 — Whitespace & case fixed')
print('Gender values now:', df_clean['gender'].unique())

In [ ]:
# ---- STEP 2: Fix invalid category values ----
valid_genders = ['male', 'female']
valid_lunch   = ['standard', 'free/reduced']

before_gender = len(df_clean)
df_clean = df_clean[df_clean['gender'].isin(valid_genders)]
df_clean = df_clean[df_clean['lunch'].isin(valid_lunch)]
df_clean.reset_index(drop=True, inplace=True)

print('STEP 2 — Invalid category rows removed')
print(f'Rows removed: {before_gender - len(df_clean)}')
print('Remaining rows:', len(df_clean))

In [ ]:
# ---- STEP 3: Fix outliers (clip scores to valid range 0-100) ----
score_cols = ['math score', 'reading score', 'writing score']

print('STEP 3 — Outliers before clipping:')
for col in score_cols:
    bad = df_clean[(df_clean[col] < 0) | (df_clean[col] > 100)][col].dropna()
    if len(bad) > 0:
        print(f'  {col}: {list(bad.values)}')

for col in score_cols:
    df_clean[col] = df_clean[col].clip(0, 100)

print('Outliers clipped to range [0, 100]')

In [ ]:
# ---- STEP 4: Handle missing values (fill with column median) ----
print('STEP 4 — Missing values before filling:')
print(df_clean[score_cols].isnull().sum())

for col in score_cols:
    median_val = df_clean[col].median()
    df_clean[col].fillna(median_val, inplace=True)
    print(f'  Filled {col} NaN with median = {median_val:.1f}')

print('\nMissing values after filling:')
print(df_clean[score_cols].isnull().sum())

In [ ]:
# ---- STEP 5: Remove duplicate rows ----
before_dedup = len(df_clean)
df_clean.drop_duplicates(inplace=True)
df_clean.reset_index(drop=True, inplace=True)

print('STEP 5 — Duplicates removed')
print(f'Rows before: {before_dedup} | Rows after: {len(df_clean)}')

## Task 3.4 — Before vs After Summary & Save

In [ ]:
print('='*50)
print('       BEFORE vs AFTER CLEANING SUMMARY')
print('='*50)
print(f'{"Metric":<28} {"Before":>10} {"After":>10}')
print('-'*50)
print(f'{"Total rows":<28} {df_messy.shape[0]:>10} {df_clean.shape[0]:>10}')
print(f'{"Total columns":<28} {df_messy.shape[1]:>10} {df_clean.shape[1]:>10}')
print(f'{"Missing values":<28} {df_messy.isnull().sum().sum():>10} {df_clean.isnull().sum().sum():>10}')
print(f'{"Duplicate rows":<28} {df_messy.duplicated().sum():>10} {df_clean.duplicated().sum():>10}')

# Save cleaned file
df_clean.to_csv('data/cleaned_students.csv', index=False)
print('\nCleaned dataset saved to data/cleaned_students.csv')

---
# TASK 4 — EDA & Correlation

## Task 4.1 — Load Cleaned Dataset

In [ ]:
df_eda = pd.read_csv('data/cleaned_students.csv')
scores = ['math score', 'reading score', 'writing score']

print('Cleaned dataset loaded for EDA')
print('Shape:', df_eda.shape)
df_eda.head()

## Task 4.2 — GroupBy Analysis

In [ ]:
# GroupBy gender
print('=== Average Scores by Gender ===')
g1 = df_eda.groupby('gender')[scores].mean().round(2)
print(g1)

In [ ]:
# GroupBy race/ethnicity
print('=== Average Scores by Race/Ethnicity ===')
g2 = df_eda.groupby('race/ethnicity')[scores].mean().round(2)
print(g2)

In [ ]:
# GroupBy parental level of education
print('=== Average Scores by Parental Level of Education ===')
g3 = df_eda.groupby('parental level of education')[scores].mean().round(2)
print(g3)

In [ ]:
# GroupBy test preparation course
print('=== Average Scores by Test Preparation Course ===')
g4 = df_eda.groupby('test preparation course')[scores].mean().round(2)
print(g4)

print()
# Score improvement from completing the course
diff = g4.loc['completed'] - g4.loc['none']
print('Score improvement from completing prep course:')
print(diff.round(2))

## Task 4.3 — Correlation Matrix & Heatmap

In [ ]:
# Correlation matrix
print('=== Correlation Matrix ===')
corr = df_eda[scores].corr()
print(corr.round(4))

In [ ]:
# Seaborn heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt='.4f',
    cmap='coolwarm',
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — Math, Reading & Writing Scores', fontsize=13, pad=15)
plt.xticks(rotation=20, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved as heatmap.png')

## Task 4.4 — Insights (300–400 words)

**EDA Insights — Student Performance in Exams**

**1. Gender Differences:** A clear pattern emerges across gender lines. Female students consistently outperform male students in reading and writing scores — by approximately 7 and 9 points respectively. However, male students hold a slight advantage in math scores, averaging about 5 points higher. This suggests that verbal and writing skills favor female students in this dataset, while quantitative performance is slightly stronger among males.

**2. Race and Ethnicity:** Students in group E consistently score the highest across all three subjects (math ~73.8, reading ~73.0, writing ~71.4), while students in group A score the lowest (math ~61.6, reading ~64.7, writing ~63.0). This could reflect underlying socioeconomic differences among groups and warrants further investigation with additional demographic variables.

**3. Parental Education Impact:** The relationship between parental education level and student performance is strongly positive. Students whose parents have a master's degree score the highest on average across all subjects, while students whose parents have only some high school education score the lowest. This highlights the important role that household educational background plays in student achievement — likely through factors like access to resources, study habits, and academic expectations at home.

**4. Test Preparation Course:** Students who completed the test preparation course show meaningful improvement across all subjects — approximately 5.6 points in math, 7.1 in reading, and 8.1 in writing compared to students who did not complete the course. This confirms that structured preparation significantly benefits academic outcomes, especially in writing tasks.

**5. Correlation Between Subjects:** Reading and writing scores show a very strong positive correlation (r ≈ 0.95), which makes intuitive sense — both are language-based skills that reinforce each other. Math has a moderate correlation with both reading (r ≈ 0.82) and writing (r ≈ 0.80), suggesting that strong general academic ability links all three subjects, though language and math skills are still somewhat distinct skill sets.

**Summary:** Overall, parental education level and test preparation course completion are the two strongest predictors of student performance in this dataset. Gender differences are present but more subject-specific. These insights can guide educational policy decisions around parental engagement programs and the expansion of test preparation resources.

---
# TASK 5 — Data Visualization

In [ ]:
# Load cleaned dataset for visualization
df_viz = pd.read_csv('data/cleaned_students.csv')
print('Dataset loaded for visualization. Shape:', df_viz.shape)

## Task 5.1 — Histogram of Math Score with KDE

In [ ]:
plt.figure(figsize=(9, 6))
sns.histplot(
    df_viz['math score'],
    bins=20,
    kde=True,
    color='steelblue',
    edgecolor='white',
    linewidth=0.5
)
plt.axvline(df_viz['math score'].mean(),   color='red',    linestyle='--', linewidth=1.5, label=f"Mean = {df_viz['math score'].mean():.1f}")
plt.axvline(df_viz['math score'].median(), color='orange', linestyle='--', linewidth=1.5, label=f"Median = {df_viz['math score'].median():.1f}")
plt.title('Distribution of Math Scores with KDE', fontsize=14, fontweight='bold')
plt.xlabel('Math Score', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('plot1_histogram_math.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

## Task 5.2 — Boxplot of Math Score by Gender

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(
    x='gender',
    y='math score',
    data=df_viz,
    palette={'male': 'steelblue', 'female': 'salmon'},
    width=0.5,
    linewidth=1.2
)
plt.title('Math Score Distribution by Gender', fontsize=14, fontweight='bold')
plt.xlabel('Gender', fontsize=12)
plt.ylabel('Math Score', fontsize=12)
plt.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('plot2_boxplot_gender.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved.')

## Task 5.3 — Scatter Plot: Reading vs Writing Score by Gender

In [ ]:
plt.figure(figsize=(9, 6))
colors = {'male': 'steelblue', 'female': 'salmon'}

for gender, group in df_viz.groupby('gender'):
    plt.scatter(
        group['reading score'],
        group['writing score'],
        label=gender.capitalize(),
        color=colors[gender],
        alpha=0.55,
        s=35,
        edgecolors='white',
        linewidth=0.3
    )

# Add regression line
m, b = np.polyfit(df_viz['reading score'], df_viz['writing score'], 1)
x_line = np.linspace(df_viz['reading score'].min(), df_viz['reading score'].max(), 100)
plt.plot(x_line, m*x_line + b, color='black', linewidth=1.2, linestyle='--', label='Trend line', alpha=0.6)

plt.title('Reading Score vs Writing Score by Gender', fontsize=14, fontweight='bold')
plt.xlabel('Reading Score', fontsize=12)
plt.ylabel('Writing Score', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('plot3_scatter_reading_writing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved.')

## Task 5.4 — Bar Chart: Average Math Score by Parental Education

In [ ]:
edu_avg = df_viz.groupby('parental level of education')['math score'].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(
    edu_avg.index,
    edu_avg.values,
    color=plt.cm.Blues(np.linspace(0.4, 0.85, len(edu_avg))),
    edgecolor='white',
    height=0.55
)

# Add value labels
for bar, val in zip(bars, edu_avg.values):
    plt.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', fontsize=10, color='black')

plt.title('Average Math Score by Parental Level of Education', fontsize=14, fontweight='bold')
plt.xlabel('Average Math Score', fontsize=12)
plt.ylabel('Parental Education Level', fontsize=12)
plt.xlim(0, 80)
plt.grid(True, axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('plot4_bar_education.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved.')

## Task 5.5 — Combined 2×2 Subplot Figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Student Performance Analysis — Combined Visualization',
             fontsize=15, fontweight='bold', y=1.01)

# ---- Plot 1: Histogram ----
ax1 = axes[0, 0]
sns.histplot(df_viz['math score'], bins=20, kde=True,
             color='steelblue', edgecolor='white', linewidth=0.5, ax=ax1)
ax1.axvline(df_viz['math score'].mean(), color='red', linestyle='--',
            linewidth=1.5, label=f"Mean={df_viz['math score'].mean():.1f}")
ax1.set_title('Math Score Distribution', fontweight='bold')
ax1.set_xlabel('Math Score')
ax1.set_ylabel('Count')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.4)

# ---- Plot 2: Boxplot ----
ax2 = axes[0, 1]
sns.boxplot(x='gender', y='math score', data=df_viz,
            palette={'male': 'steelblue', 'female': 'salmon'},
            width=0.5, linewidth=1.2, ax=ax2)
ax2.set_title('Math Score by Gender', fontweight='bold')
ax2.set_xlabel('Gender')
ax2.set_ylabel('Math Score')
ax2.grid(True, axis='y', alpha=0.4)

# ---- Plot 3: Scatter ----
ax3 = axes[1, 0]
for gender, group in df_viz.groupby('gender'):
    ax3.scatter(group['reading score'], group['writing score'],
                label=gender.capitalize(), color=colors[gender],
                alpha=0.5, s=20, edgecolors='white', linewidth=0.3)
ax3.set_title('Reading vs Writing Score', fontweight='bold')
ax3.set_xlabel('Reading Score')
ax3.set_ylabel('Writing Score')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.4)

# ---- Plot 4: Bar chart ----
ax4 = axes[1, 1]
bars = ax4.barh(edu_avg.index, edu_avg.values,
                color=plt.cm.Blues(np.linspace(0.4, 0.85, len(edu_avg))),
                edgecolor='white', height=0.55)
for bar, val in zip(bars, edu_avg.values):
    ax4.text(val + 0.2, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', fontsize=8)
ax4.set_title('Avg Math Score by Parental Education', fontweight='bold')
ax4.set_xlabel('Average Math Score')
ax4.set_ylabel('Education Level')
ax4.set_xlim(0, 80)
ax4.grid(True, axis='x', alpha=0.4)
ax4.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('best_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Combined figure saved as best_visualization.png')

---
## All Tasks Complete!

**Files produced:**
- `data/messy_students.csv` — messy dataset (Task 3)
- `data/cleaned_students.csv` — cleaned dataset (Task 3)
- `heatmap.png` — correlation heatmap (Task 4)
- `plot1_histogram_math.png` (Task 5)
- `plot2_boxplot_gender.png` (Task 5)
- `plot3_scatter_reading_writing.png` (Task 5)
- `plot4_bar_education.png` (Task 5)
- `best_visualization.png` — combined 2x2 figure (Task 5)

**Next step:** Take screenshots of all code + outputs, then compile the final PDF.